[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/26.%20The%20Gate%20Automation%20%26%20Damage%20Detection%20Problem/P26-Tier-2-CLEAN_executed.ipynb)

---


# 26. The Gate Automation and Damage Detection Problem

## Tier 2: A Priority-Based Greedy Heuristic

### Goal and outputs

This notebook implements a priority-based greedy heuristic for allocating a limited hourly monitoring budget among gate sensors. At each hour, the algorithm scores every sensor, considers sensors in descending priority order, and selects each sensor that still fits within the remaining budget. A simulation then updates sensor and gate states over the planning horizon.

The main outputs are the hourly monitoring schedule, system reliability, budget utilization, and gate availability. The example uses four gates, twelve sensors, 48 hourly periods, and a budget of $200 per hour. These are inputs to the example, not guaranteed performance results.

### Scope and modeling assumptions

- The priority score is a weighted heuristic, not a probability of failure or a monetary benefit.
- The budget is enforced independently in each hour. Monitoring costs are treated as fixed by sensor type.
- Each sensor is considered independently for selection. The greedy rule does not reconsider earlier choices to improve the combination of selected sensors.
- Sensor health changes according to simple monitoring and environmental-stress rules. Failure events are stochastic and use Python's global random-number generator; no random seed is set, so simulation results can vary between runs.
- A gate is operational when the mean health of its sensors is at least 0.2. System reliability is the percentage of gates operational after each hourly state update.
- Downtime costs are included in the instance data but are not used in the priority score, budget rule, state transitions, or reported objective. Vehicle throughput is accumulated for operational gates, but it does not affect sensor selection.
- The notebook evaluates the heuristic by simulation. It does not implement or compare against an optimal solution, so it does not provide an optimality gap or an optimality guarantee.

### Solution sequence

1. Define the data structures and example instance.
2. Calculate a weighted priority score for each sensor and hour.
3. Use a priority queue to apply the budget-constrained greedy selection rule.
4. Update sensor and gate states and record simulation metrics.
5. Summarize and visualize the results, then discuss computational complexity and the heuristic's limitations.

In [ ]:
import heapq
import random
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np

print("Priority-Based Sensor Monitoring Algorithm")
print("=" * 50)

### Data Structures

A data structure gives each part of the model a clear representation in code. `SensorState` and `GateState` hold quantities that change during the simulation. `HeuristicInstance` groups the input data, while `PrioritySensor` is the item placed in the priority queue.

The priority queue uses Python's `heapq`, which is a min-heap by default. `PrioritySensor.__lt__` reverses the comparison so that the item with the numerically largest priority is removed first. This ordering implements the heuristic's rule to consider the highest-scoring sensor first.

In [ ]:
@dataclass
class SensorState:
    gate_id: int
    sensor_type: str
    health_index: float = 1.0
    failure_count: int = 0
    environmental_factor: float = 1.0


@dataclass
class GateState:
    gate_id: int
    demand_level: float = 1.0
    is_operational: bool = True
    total_vehicles_processed: int = 0


@dataclass
class PrioritySensor:
    priority: float
    gate_id: int
    sensor_type: str

    def __lt__(self, other):
        # Reverse the default min-heap ordering so the highest score is selected first.
        return self.priority > other.priority


@dataclass
class HeuristicInstance:
    gates: List[GateState]
    sensors: List[SensorState]
    time_periods: int
    monitoring_budget_per_hour: float
    sensor_costs: Dict[str, float]
    gate_downtime_costs: Dict[int, float]
    demand_patterns: Dict[int, List[float]]

### Create the Example Instance

The instance supplies the data used by the scoring rule and simulation: sensor costs, gate downtime costs, hourly demand patterns, initial gate states, and initial sensor states. Demand has recurring higher periods during the morning and evening. The parameters below define the example; they do not imply a particular reliability outcome.

In [ ]:
def create_heuristic_example():
    """Build the four-gate, 48-hour sensor-monitoring example."""
    sensor_costs = {
        "photo_eye": 50,
        "ground_loop": 60,
        "safety_beam": 70,
    }
    gate_downtime_costs = {1: 15000, 2: 12000, 3: 10000, 4: 8000}

    # Demand multipliers distinguish peak, shoulder, and off-peak hours.
    demand_patterns = {}
    for gate_id in range(1, 5):
        base_demand = 150 - (gate_id - 1) * 25
        pattern = []
        for hour in range(48):
            hour_of_day = hour % 24
            if 8 <= hour_of_day <= 10 or 17 <= hour_of_day <= 19:
                peak_factor = 1.8
            elif 6 <= hour_of_day <= 11 or 16 <= hour_of_day <= 21:
                peak_factor = 1.3
            else:
                peak_factor = 0.7
            pattern.append(base_demand * peak_factor)
        demand_patterns[gate_id] = pattern

    gates = []
    for gate_id in range(1, 5):
        gates.append(
            GateState(
                gate_id=gate_id,
                demand_level=demand_patterns[gate_id][0] / 100,
            )
        )

    sensor_configs = [
        (1, "photo_eye", 0.95, 5, 1.2),
        (1, "ground_loop", 0.85, 2, 1.0),
        (1, "safety_beam", 0.90, 1, 1.1),
        (2, "photo_eye", 0.88, 3, 1.0),
        (2, "ground_loop", 0.92, 1, 1.3),
        (2, "safety_beam", 0.87, 4, 1.1),
        (3, "photo_eye", 0.93, 1, 0.9),
        (3, "ground_loop", 0.89, 2, 1.0),
        (3, "safety_beam", 0.91, 1, 0.8),
        (4, "photo_eye", 0.94, 0, 0.8),
        (4, "ground_loop", 0.96, 0, 0.9),
        (4, "safety_beam", 0.95, 1, 0.7),
    ]

    sensors = [
        SensorState(
            gate_id=gate_id,
            sensor_type=sensor_type,
            health_index=health,
            failure_count=failures,
            environmental_factor=env_factor,
        )
        for gate_id, sensor_type, health, failures, env_factor in sensor_configs
    ]

    return HeuristicInstance(
        gates=gates,
        sensors=sensors,
        time_periods=48,
        monitoring_budget_per_hour=200,
        sensor_costs=sensor_costs,
        gate_downtime_costs=gate_downtime_costs,
        demand_patterns=demand_patterns,
    )


heuristic_instance = create_heuristic_example()
print("Heuristic instance created:")
print(f"  {len(heuristic_instance.gates)} gates")
print(f"  {len(heuristic_instance.sensors)} sensors")
print(f"  {heuristic_instance.time_periods} time periods")
print(
    "  Budget: "
    f"${heuristic_instance.monitoring_budget_per_hour}/hour"
)

### Priority-Based Algorithm Implementation

#### Priority score

The score combines five indicators: low sensor health, failure history, current gate demand, environmental stress, and sensor criticality. A higher score means that the heuristic considers a sensor more important to monitor; it is not a calibrated risk estimate.

For sensor $s$ at gate $g$ in hour $t$, the implementation calculates

$$
P_{s,t} = w_h(1 - H_s)
+ w_f \min\!\left(\frac{F_s}{10}, 1\right)
+ w_d \frac{D_{g,t}}{150}
+ w_e \frac{E_s - 0.5}{1.5}
+ w_c \frac{C_s}{3}.
$$

Here, $H_s$ is the sensor's health index, $F_s$ is its failure count, $D_{g,t}$ is gate demand, $E_s$ is the environmental factor, and $C_s$ is the sensor-type criticality value. The weights are $w_h=0.30$, $w_f=0.25$, $w_d=0.20$, $w_e=0.15$, and $w_c=0.10$. The code represents these values in `priority_weights` and `sensor_criticality`; `calculate_priority_score` implements the equation.

#### Budget-constrained greedy selection

Let $S$ be the set of sensors, $c_s$ the hourly cost of monitoring sensor $s$, and $B$ the hourly budget. The algorithm orders sensors by decreasing $P_{s,t}$. Starting with remaining budget $R=B$, it considers each sensor in that order and selects it if $c_s \leq R$, then reduces $R$ by $c_s$. A sensor that does not fit is skipped, and the algorithm continues to the next sensor. Thus, the selection is feasible for the budget, but the rule does not guarantee the best-value combination of sensors.

#### State updates and reliability

If a sensor is monitored, its health increases by 0.01 up to 1, and its failure probability is $0.005E_s$. Otherwise, its health decreases by $0.02E_s$ down to 0, and its failure probability is $0.02E_s$. A failure event increments the failure count and reduces health by a further 0.1, but only when the updated health is below 0.3. These probabilities are evaluated with random draws.

A gate is marked operational if its sensors' mean health is at least 0.2. If $G$ is the set of gates and $O_t$ is the operational-gate subset after the update in hour $t$, the reported reliability is

$$
\text{Reliability}_t = 100 \times \frac{|O_t|}{|G|}.
$$

This is an availability measure for the modeled gates, not a probability that the complete terminal system will function.

In [ ]:
class PriorityBasedSensorMonitoring:
    def __init__(self, instance: HeuristicInstance):
        self.instance = instance
        self.priority_weights = {
            "health": 0.3,
            "failure_history": 0.25,
            "demand": 0.2,
            "environmental": 0.15,
            "criticality": 0.1,
        }
        self.sensor_criticality = {
            "safety_beam": 3.0,
            "photo_eye": 2.0,
            "ground_loop": 1.0,
        }
        self.results = {
            "monitoring_schedule": [],
            "system_reliability": [],
            "budget_utilization": [],
            "gate_availabilities": [],
        }

    def calculate_priority_score(
        self, sensor: SensorState, gate: GateState, hour: int
    ) -> float:
        """Calculate a sensor's weighted monitoring priority for one hour."""
        health_score = (
            (1.0 - sensor.health_index) * self.priority_weights["health"]
        )
        failure_score = (
            min(sensor.failure_count / 10.0, 1.0)
            * self.priority_weights["failure_history"]
        )
        current_demand = (
            self.instance.demand_patterns[gate.gate_id][hour] / 150.0
        )
        demand_score = current_demand * self.priority_weights["demand"]
        environmental_score = (
            (sensor.environmental_factor - 0.5)
            / 1.5
            * self.priority_weights["environmental"]
        )
        criticality_score = (
            self.sensor_criticality[sensor.sensor_type]
            / 3.0
            * self.priority_weights["criticality"]
        )

        return (
            health_score
            + failure_score
            + demand_score
            + environmental_score
            + criticality_score
        )

    def select_sensors_to_monitor(self, hour: int) -> List[Tuple[int, str]]:
        """Apply the priority-ordered greedy rule under the hourly budget."""
        priority_queue = []

        for sensor in self.instance.sensors:
            gate = self.instance.gates[sensor.gate_id - 1]
            priority = self.calculate_priority_score(sensor, gate, hour)
            heapq.heappush(
                priority_queue,
                PrioritySensor(
                    priority=priority,
                    gate_id=sensor.gate_id,
                    sensor_type=sensor.sensor_type,
                ),
            )

        selected_sensors = []
        remaining_budget = self.instance.monitoring_budget_per_hour

        while priority_queue and remaining_budget > 0:
            priority_sensor = heapq.heappop(priority_queue)
            sensor_cost = self.instance.sensor_costs[
                priority_sensor.sensor_type
            ]

            if sensor_cost <= remaining_budget:
                selected_sensors.append(
                    (priority_sensor.gate_id, priority_sensor.sensor_type)
                )
                remaining_budget -= sensor_cost

        return selected_sensors

    def update_states(
        self, monitored_sensors: List[Tuple[int, str]], hour: int
    ) -> None:
        """Update sensor health, stochastic failures, and gate states."""
        monitored_set = set(monitored_sensors)

        for sensor in self.instance.sensors:
            is_monitored = (sensor.gate_id, sensor.sensor_type) in monitored_set

            if is_monitored:
                sensor.health_index = min(1.0, sensor.health_index + 0.01)
                failure_probability = 0.005 * sensor.environmental_factor
            else:
                degradation = 0.02 * sensor.environmental_factor
                sensor.health_index = max(0.0, sensor.health_index - degradation)
                failure_probability = 0.02 * sensor.environmental_factor

            if (
                random.random() < failure_probability
                and sensor.health_index < 0.3
            ):
                sensor.failure_count += 1
                sensor.health_index = max(0.0, sensor.health_index - 0.1)

        for gate in self.instance.gates:
            gate.demand_level = (
                self.instance.demand_patterns[gate.gate_id][hour] / 100.0
            )
            gate_sensors = [
                sensor
                for sensor in self.instance.sensors
                if sensor.gate_id == gate.gate_id
            ]
            average_health = np.mean(
                [sensor.health_index for sensor in gate_sensors]
            )
            gate.is_operational = average_health >= 0.2

            if gate.is_operational:
                vehicles_processed = int(
                    self.instance.demand_patterns[gate.gate_id][hour]
                )
                gate.total_vehicles_processed += vehicles_processed

    def run_simulation(self) -> Dict:
        """Run the hourly selection and state-update sequence."""
        print("Running the priority-based sensor monitoring simulation...")

        for hour in range(self.instance.time_periods):
            monitored_sensors = self.select_sensors_to_monitor(hour)
            monitoring_cost = sum(
                self.instance.sensor_costs[sensor_type]
                for _, sensor_type in monitored_sensors
            )
            assert monitoring_cost <= self.instance.monitoring_budget_per_hour

            self.update_states(monitored_sensors, hour)
            self.results["monitoring_schedule"].append(monitored_sensors)

            operational_gates = sum(
                1 for gate in self.instance.gates if gate.is_operational
            )
            reliability = operational_gates / len(self.instance.gates) * 100
            self.results["system_reliability"].append(reliability)

            budget_utilization = (
                monitoring_cost
                / self.instance.monitoring_budget_per_hour
                * 100
            )
            self.results["budget_utilization"].append(budget_utilization)

            gate_availability = {
                gate.gate_id: (100 if gate.is_operational else 0)
                for gate in self.instance.gates
            }
            self.results["gate_availabilities"].append(gate_availability)

            if (hour + 1) % 12 == 0:
                print(
                    f"  Hour {hour + 1:2d}/{self.instance.time_periods}: "
                    f"Reliability = {reliability:.1f}%"
                )

        return self.results

### Execute the Algorithm

The simulation applies the same sequence in every period: select sensors using the current state and demand, update sensor and gate states, and record the resulting measures. The reported averages summarize this one stochastic simulation run. Because there is no fixed random seed, separate runs need not produce identical results. Create a fresh instance before another run if the intention is to simulate a new 48-hour period from the initial conditions.

In [ ]:
pbsm = PriorityBasedSensorMonitoring(heuristic_instance)
results = pbsm.run_simulation()

final_reliability = np.mean(results["system_reliability"])
final_budget_utilization = np.mean(results["budget_utilization"])
total_vehicles = sum(
    gate.total_vehicles_processed for gate in heuristic_instance.gates
)

print("\nSIMULATION RESULTS")
print(f"Average system reliability: {final_reliability:.1f}%")
print(f"Average budget utilization: {final_budget_utilization:.1f}%")
print(f"Total vehicles processed: {total_vehicles:,}")
print(f"Simulation duration: {heuristic_instance.time_periods} hours")

assert len(results["system_reliability"]) == heuristic_instance.time_periods
assert len(results["budget_utilization"]) == heuristic_instance.time_periods
assert all(
    0 <= utilization <= 100
    for utilization in results["budget_utilization"]
)


### Visualization and Analysis

The panels show different aspects of the simulated schedule. Hourly system reliability and budget utilization are time series. Monitoring frequency reports how often each sensor was selected as a percentage of the full planning horizon. Average gate availability is the fraction of simulated hours in which each gate was operational. These plots describe the simulated run; they do not establish that the heuristic is optimal or that the model predicts real-world performance.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Priority-Based Sensor Monitoring Results",
    fontsize=14,
    fontweight="bold",
)

# Plot reliability after each hourly state update.
ax1 = axes[0, 0]
hours = list(range(len(results["system_reliability"])))
reliability = results["system_reliability"]
ax1.plot(hours, reliability, linewidth=2, color="#2ecc71")
ax1.fill_between(hours, reliability, alpha=0.3, color="#2ecc71")
ax1.set_title("System Reliability Over Time")
ax1.set_xlabel("Hour")
ax1.set_ylabel("Operational Gates (%)")
ax1.set_ylim(0, 105)
ax1.grid(True, alpha=0.3)

# Compare hourly monitoring expenditure with the budget limit.
ax2 = axes[0, 1]
budget_utilization = results["budget_utilization"]
ax2.plot(hours, budget_utilization, linewidth=2, color="#3498db")
ax2.fill_between(hours, budget_utilization, alpha=0.3, color="#3498db")
ax2.axhline(
    y=100,
    color="red",
    linestyle="--",
    alpha=0.7,
    label="Hourly budget limit",
)
ax2.set_title("Budget Utilization")
ax2.set_xlabel("Hour")
ax2.set_ylabel("Budget Used (%)")
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)
ax2.legend()

# Count selections across the horizon and display up to eight sensors.
ax3 = axes[1, 0]
monitoring_counts = defaultdict(int)
for hour_schedule in results["monitoring_schedule"]:
    for gate_id, sensor_type in hour_schedule:
        monitoring_counts[(gate_id, sensor_type)] += 1

monitoring_percentages = {
    sensor: count / heuristic_instance.time_periods * 100
    for sensor, count in monitoring_counts.items()
}
sorted_sensors = sorted(
    monitoring_percentages.items(),
    key=lambda item: item[1],
    reverse=True,
)
top_sensors = sorted_sensors[:8]
sensor_labels = [
    f"Gate {gate_id} - {sensor_type}"
    for (gate_id, sensor_type), _ in top_sensors
]
sensor_values = [percentage for _, percentage in top_sensors]
ax3.bar(sensor_labels, sensor_values, color="#9b59b6")
ax3.set_title("Most Frequently Monitored Sensors (Up to 8)")
ax3.set_xlabel("Sensor")
ax3.set_ylabel("Hours Monitored (% of horizon)")
ax3.set_xticks(range(len(sensor_labels)))
ax3.set_xticklabels(sensor_labels, rotation=45, ha="right")
ax3.grid(axis="y", alpha=0.3)

# Each hourly availability value is either 0% or 100% for a gate.
ax4 = axes[1, 1]
gate_availabilities = {
    gate.gate_id: [] for gate in heuristic_instance.gates
}
for hour_data in results["gate_availabilities"]:
    for gate_id, availability in hour_data.items():
        gate_availabilities[gate_id].append(availability)

gate_ids = sorted(gate_availabilities)
average_availabilities = [
    np.mean(gate_availabilities[gate_id]) for gate_id in gate_ids
]
gate_labels = [f"Gate {gate_id}" for gate_id in gate_ids]
gate_colors = [
    "#2ecc71" if availability > 95
    else "#f39c12" if availability > 90
    else "#e74c3c"
    for availability in average_availabilities
]
bars = ax4.bar(gate_labels, average_availabilities, color=gate_colors)
ax4.set_title("Average Gate Availability")
ax4.set_xlabel("Gate")
ax4.set_ylabel("Operational Hours (% of horizon)")
ax4.set_ylim(0, 105)
ax4.grid(axis="y", alpha=0.3)

for bar, availability in zip(bars, average_availabilities):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{availability:.1f}%",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()
print("Simulation visualizations completed.")

### Performance Analysis

Let $S$ be the number of sensors, $G$ the number of gates, and $T$ the number of hourly periods. For one period, the implementation scores each sensor in constant time and inserts it into a binary heap. It may then remove every sensor from the heap while applying the budget rule. Therefore, the worst-case time is $O(S\log S + G)$ per period: heap insertion and removal contribute $O(S\log S)$, while updating gate states requires scanning the sensors by gate and is bounded by $O(S+G)$. Across the full horizon, the simulation takes $O(T(S\log S + G))$ time.

The priority queue for one period uses $O(S)$ temporary space. Stored schedules and gate availability records use $O(TS + TG)$ space in the worst case. The demand-pattern input also occupies $O(TG)$ space. These bounds describe how work and storage grow with instance size; they are not measured runtime projections. The greedy method has no optimality guarantee, and its result depends on the score weights, data, budget, and stochastic state updates.

### Why This Tier Exists Alongside an Optimization-Based Tier

This tier demonstrates a direct decision rule: assign scores, order candidates, and select feasible sensors until the budget can no longer accommodate the next candidate. Its computational work is driven primarily by sorting behavior implemented through the priority queue, making it suitable for illustrating a fast, sequential allocation heuristic.

The score combines several considerations in one ranking, but the weights encode modeling choices rather than a proof of value. The greedy selection is order-dependent: once a sensor is selected, the procedure does not exchange it for a different combination. It can therefore return a budget-feasible schedule without maximizing a separately specified objective. In this notebook, there is no objective-based optimality comparison.

The approach is useful when a transparent priority rule and rapid decisions are desired. Its limitations include sensitivity to weights, lack of an optimality guarantee, simplified sensor and gate dynamics, and limited coordination across hours. Further modeling could represent additional operational constraints or calibrate the scoring inputs using domain data; those extensions are outside the implementation in this notebook.